In [4]:
import math
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from collections import Counter
from torch.nn.utils.rnn import pad_sequence
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


### Loading Dataset

In [5]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-train.en", "r", encoding="utf-8") as f:
    english_train = [line.strip() for line in f.readlines()]
english_train[:5]

['Other, Private Use',
 '[SCREAMING]',
 'Spouse',
 'I will never salute you!',
 'and the stars and the trees bow themselves;']

In [6]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-train.hi", "r", encoding="utf-8") as f:
    hindi_train =[line.strip() for line in f.readlines()]
hindi_train[:5]

['अन्य, निज़ी उपयोग',
 'ऊबड़ .',
 'जीवनसाथी',
 '- तुम एक कमांडर कभी नहीं होगा!',
 'और तारे और वृक्ष सजदा करते है;']

In [7]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-dev.en", "r", encoding="utf-8") as f:
    english_val = [line.strip() for line in f.readlines()]
english_val[:5]

['No, no, not so fast.',
 ', eject!',
 "I'm Dr. Messa.",
 'So we notify the cops about big ticket sales and we even keep half a dozen Ukrainian ex-naval commandos in a van outside, just in case it all kicks off.',
 'receiving what their Lord has given them, for they had been virtuous aforetime.']

In [8]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-dev.hi", "r", encoding="utf-8") as f:
    hindi_val = [line.strip() for line in f.readlines()]
hindi_val[:5]

['तुम इतनी आसानी से छूट नहीं सकते.',
 ', बेदखल!',
 'Messa हूँ.',
 'तोहमबड़ीटिकटोंकीबिक्रीकेबारे मेंपुलिस सूचित... / मैं ... और हम भी रखना आधा दर्जन यूक्रेनी पूर्व नौसेना कमांडो...',
 'जो कुछ उनके रब ने उन्हें दिया, वे उसे ले रहे होंगे। निस्संदेह वे इससे पहले उत्तमकारों में से थे']

In [9]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-test.en", "r", encoding="utf-8") as f:
    english_test = [line.strip() for line in f.readlines()]
english_test[:5]

['Give shots of injections or pills, but he must be alright soon.',
 'They said, “O Shuaib, we do not understand much of what you say, and we see that you are weak among us. Were it not for your tribe, we would have stoned you. You are of no value to us.”',
 '- Yeah.',
 'If evil befalls him he is perturbed;',
 '♪ BE FOREVER BOUND']

In [10]:
with open(r"C:\Users\milin\OneDrive\Desktop\Cerebral Cinema\english to hindi traslation dataset (1)\hindi to english dataset\opus.en-hi-test.hi", "r", encoding="utf-8") as f:
    hindi_test = [line.strip() for line in f.readlines()]
hindi_test[:5]

['सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.',
 'और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो उनमें से अक्सर तो हमारी समझ ही में नहीं आयी और इसमें तो शक नहीं कि हम तुम्हें अपने लोगों में बहुत कमज़ोर समझते है और अगर तुम्हारा क़बीला न होता तो हम तुम को (कब का) संगसार कर चुके होते और तुम तो हम पर किसी तरह ग़ालिब नहीं आ सकते',
 '- हाँ.',
 'जि उसे तकलीफ़ पहुँचती है तो घबरा उठता है,',
 '♪हमेशाके लिएबाध्यहोने']

In [11]:
def tokenize(text):
    return text.lower().split()

### Build Vocabulary

In [19]:
class Vocabulary:
    # Two dictionaries are maintained.
    def __init__(self):
        
        self.word2idx={
            '<pad>':0, # padding
            '<unk>':1, # unknown word 
            '<bos>':2, # beginning of sentence 
            '<eos>':3  # end of sentence 
        } # word->int

        self.idx2word={
            0: '<pad>',
            1: '<unk>',
            2: '<bos>',
            3: '<eos>'
        } # int->word

        self.n_words = 4

    def build_vocab(self, sentences):

        for sentence in sentences:

            tokens=tokenize(sentence)

            for token in tokens:
                if token not in self.word2idx:

                    self.word2idx[token]=self.n_words
                    self.idx2word[self.n_words]=token
                    self.n_words +=1
    def numericalize(self,sentence):
        tokens=tokenize(sentence)
        return [self.word2idx.get(token,self.word2idx['<unk>']) for token in tokens]

In [20]:
ENG_VOCAB=Vocabulary()
ENG_VOCAB.build_vocab(english_train)

HIN_VOCAB=Vocabulary()
HIN_VOCAB.build_vocab(hindi_train)

In [21]:
print("English Vocabulary Size:", ENG_VOCAB.n_words)
print("Hindi Vocabulary Size:", HIN_VOCAB.n_words)

English Vocabulary Size: 136306
Hindi Vocabulary Size: 109481


In [22]:
def encode(sentence, vocabulary):

    tokens = vocabulary.numericalize(sentence)

    ids = [vocabulary.word2idx['<bos>']]

    ids += tokens

    ids += [vocabulary.word2idx['<eos>']]

    return torch.tensor(ids)